# Finding Redundant 1D Subspaces in LLM Activations

This notebook discovers **redundant 1-dimensional subspaces** in language model activations: directions where:
1. Activations have **large projections** (model actively uses this direction)
2. **Removing** this projection has **minimal impact** on output token distributions

In [1]:
import sys
import pathlib

# set pythonpath to the main module directory
module_dir = pathlib.Path("..").parent.resolve().parent
if str(module_dir) not in sys.path:
    sys.path.append(str(module_dir))

In [2]:
from scripts.utils.load_dataset import DatasetName, load_dataset

ds = load_dataset(DatasetName.OASST2)

INFO 03-01 21:32:32 [scripts.utils.load_dataset:158] Loading dataset from disk at /home/fre.gilad/source/silent_direction/scripts/utils/../../data/oasst2


In [3]:
d = ds[0]
d

,prompt
0,[{'content': 'Could you provide a brief descri...
1,"[{'content': 'Okay, so picture this. Technolog..."
2,[{'content': 'Why do humans find music (melody...
3,"[{'content': 'How can I make replicants', 'rol..."
4,[{'content': 'Is it possible that someday ever...
...,...
14995,[{'content': 'what happens when you mix potass...
14996,[{'content': 'Why might a person prefer tea to...
14997,[{'content': 'Transform this set of items into...
14998,[{'content': '$2.5 Million / 4MWh in Euros/kWh...


In [4]:
import torch
from src.utils.env import set_seed

set_seed(42)

# torch.set_float32_matmul_precision("high")

In [5]:
import torch
from scripts.utils.load_model import load_model
from src.model import TargetedModel
from src.aliases import Conv

model_name = "microsoft/Phi-3-mini-4k-instruct"

print(f"====== Testing model {model_name} ======")
model, tokenizer = load_model(model_name)
targeted_model = TargetedModel(model, tokenizer, is_chat=True)
print(f"Model {model_name}")
print(model)

====== Testing model microsoft/Phi-3-mini-4k-instruct ======


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
INFO 03-01 21:32:52 [scripts.utils.load_model:89] Disabled gradients for all model parameters and set model to eval mode.


Model microsoft/Phi-3-mini-4k-instruct
Phi3ForCausalLM(
  (model): Phi3Model(
    (embed_tokens): Embedding(32064, 3072, padding_idx=32000)
    (layers): ModuleList(
      (0-31): 32 x Phi3DecoderLayer(
        (self_attn): Phi3Attention(
          (o_proj): Linear(in_features=3072, out_features=3072, bias=False)
          (qkv_proj): Linear(in_features=3072, out_features=9216, bias=False)
        )
        (mlp): Phi3MLP(
          (gate_up_proj): Linear(in_features=3072, out_features=16384, bias=False)
          (down_proj): Linear(in_features=8192, out_features=3072, bias=False)
          (activation_fn): SiLUActivation()
        )
        (input_layernorm): Phi3RMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): Phi3RMSNorm((3072,), eps=1e-05)
        (resid_attn_dropout): Dropout(p=0.0, inplace=False)
        (resid_mlp_dropout): Dropout(p=0.0, inplace=False)
      )
    )
    (norm): Phi3RMSNorm((3072,), eps=1e-05)
    (rotary_emb): Phi3RotaryEmbedding()
  )
  (lm_hea

In [6]:
print(model.dtype)

torch.bfloat16


In [7]:
from scripts.utils.load_dataset import load_dataset
from src.data import TableLoader

ds_train, ds_val, ds_test = load_dataset("hh-rlhf")

dl_train = TableLoader(ds_train, batch_size=16, shuffle=True)
dl_val = TableLoader(ds_val, batch_size=8, shuffle=True)
dl_test = TableLoader(ds_test, batch_size=8, shuffle=True)

INFO 03-01 21:32:52 [scripts.utils.load_dataset:75] Loading dataset from disk at /home/fre.gilad/source/silent_direction/scripts/utils/../../data/hh-rlhf


In [14]:
math_prompt_requires_reasoning = "What is the integral of the following complex function: f(x) = e^(x^2) * sin(x)?"

In [17]:
response = targeted_model.generate(
    prompts=[[{"role": "user", "content": math_prompt_requires_reasoning}]]
)

In [18]:
print(response)

["The integral of the function f(x) = e^(x^2) * sin(x) cannot be expressed in terms of elementary functions. However, we can represent it using a special function called the Fresnel integral.\n\nLet's denote the integral as I:\n\nI = ∫ e^(x^2) * sin(x) dx\n\nTo solve this integral, we can use the Fresnel S integral, which is defined as:\n\n"]
